In [1]:
from langchain_classic.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.prompts import PromptTemplate
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableSequence
from langchain_classic.document_loaders import TextLoader

In [ ]:
#Loading the document

loader = TextLoader("langchain_crewai_dataset.txt")
raw_document = loader.load()
raw_document

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content="LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)\nAt the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)\nLangChain integrates seamlessly with vector databases like FAISS, Chroma, Pinecone, and Weaviate, enabling semantic search within large document 

In [3]:
#Splitting the document 
splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
chunks = splitter.split_documents(raw_document)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, p

In [4]:
#Now embbed the chunks

embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents( chunks, embedder)
vector_store

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6441.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7}
)

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model(model="groq:llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000218FA7D8190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000218FA7D8B90>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [13]:
#Query Decomposition
decomposition_prompt = PromptTemplate.from_template(
    """You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions: """
)

In [14]:
#Create decomposition chain
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [16]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})
print(decomposition_question)

To better understand the differences and similarities between LangChain and CrewAI, I'll decompose the complex question into the following sub-questions:

1. **What is LangChain's architecture and how does it utilize memory in its workflow?**

   This sub-question will help us understand the memory component of LangChain and how it's structured.

2. **What is CrewAI's architecture and how does it utilize memory and agents in its workflow?**

   This sub-question will give us a similar understanding of CrewAI's memory and agent components.

3. **How do LangChain's memory management and agent interactions differ from CrewAI's?**

   This sub-question will help us identify the key differences between the two systems.

4. **What is the purpose of using agents in LangChain and CrewAI, and how do they contribute to the overall workflow?**

   This sub-question will provide insight into the role of agents in both systems and how they're utilized.


In [24]:
#QA chain per sub question
qa_prompt = PromptTemplate.from_template(
    """ Use the context below to answer the question.

Context:
{context}

Question: {input} 
answer"""
)

In [25]:
qa_chain = create_stuff_documents_chain(
    llm ,
    qa_prompt
)


In [26]:
#Create a function that retrieve the documents for each sub questions

def rag_pipleline_chain(user_query):
    #Decompose the query
    decomposed_query = decomposition_chain.invoke({"question" : query})

    sub_questions  = [q.strip("-•1234567890. ").strip() for q in decomposed_query.split("\n") if q.strip()]
    results = []

    for individual_question in sub_questions:
        document = retriever.invoke(individual_question)
        result = qa_chain.invoke(
            {
                "context" : document,
            "input" : individual_question 
            }
        )
        results.append(f"Q: {individual_question}\nA: {result}")
    return "\n\n".join(results)
        

In [27]:
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = rag_pipleline_chain(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: To break down the complex question into smaller sub-questions for better document retrieval, I would suggest the following:
A: It seems like you're trying to write a response based on the provided context. Here's a possible completion of your suggestion:

To break down the complex question into smaller sub-questions for better document retrieval, I would suggest the following:

1. **Exact term matching**: Use keyword-based retrieval methods like BM25 to catch exact term matches in the document corpus.
2. **Semantic search**: Utilize embedding-based retrieval methods to catch semantically similar content within the document corpus.
3. **Hybrid retrieval**: Combine the above two approaches (exact term matching and semantic search) to achieve better recall and catch both exact term matches and semantically similar content.
4. **Integration with vector databases**: Leverage vector databases like FAISS, Chroma, Pinecone, and Weaviate to enable semantic search within larg